# Diagnostic 3b: fully frozen CGM+Time predictor + image-only residual (ResNet18, dim=4)

In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

Mounted at /content/drive
IN_COLAB=True  REPO_ROOT=/content/drive/MyDrive/dissertation


In [2]:
import json

import numpy as np
import pandas as pd
import torch

from src import dataloader as data
from src import metrics
from src import train
from src.models.classical_baseline import HORIZONS
from src.models.multimodal_residual import load_frozen_baseline, FrozenImageOnlyResidualPredictor

MODEL_NAME = 'Diagnostic 3b: frozen CGM+time + ResNet18 image-only residual (dim=4)'
SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]

ENCODER_NAME = 'resnet18'
IMG_FEAT_DIM = 512
IMG_PROJ_DIM = 4

EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '4_diagnostic'
EXP2_CHECKPOINT_DIR = (
    PROJECT_ROOT / 'experiments' / '3_multimodal' / 'multimodal' / 'checkpoints' / 'gated_cgm_time'
)
RESULTS_DIR = EXPERIMENT_DIR / 'results' / 'diagnostic_3b'
SUMMARY_PATH = RESULTS_DIR / 'diagnostic3b_resnet18_dim4_validation_summary.json'
PAIRED_PATH = RESULTS_DIR / 'diagnostic3b_resnet18_dim4_paired_validation.csv'

print('MODEL_NAME:', MODEL_NAME)
print('IMG_PROJ_DIM:', IMG_PROJ_DIM)


MODEL_NAME: Diagnostic 3b: frozen CGM+time + ResNet18 image-only residual (dim=4)
IMG_PROJ_DIM: 4


In [3]:
stats = data.load_cgm_stats()

train_loader = data.load_multimodal_tensors(
    'train',
    encoder_name=ENCODER_NAME,
)
val_loader = data.load_multimodal_tensors(
    'val',
    encoder_name=ENCODER_NAME,
    shuffle=False,
)

print('n_train_batches:', len(train_loader))
print('n_val_batches:', len(val_loader))

sample_batch = next(iter(train_loader))
print('batch shapes:', [tuple(x.shape) for x in sample_batch])


n_train_batches: 323
n_val_batches: 64
batch shapes: [(256, 24), (256,), (256,), (256, 512), (256, 5)]


In [4]:
def forward_fn(model, batch, device):
    cgm, tsm, has_meal, img_feat, target = batch
    pred = model(
        cgm.to(device),
        tsm.to(device),
        img_feat.to(device),
        has_meal.to(device),
    )
    return pred, target.to(device)


In [5]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = []

for seed in SEEDS:
    print(f"\n{'-' * 18} seed={seed} {'-' * 18}")

    # Load the seed-matched trained Experiment 2 baseline.
    base_model = load_frozen_baseline(seed, EXP2_CHECKPOINT_DIR)

    # Set the seed before constructing the trainable image residual branch.
    train.set_seed(seed)

    model = FrozenImageOnlyResidualPredictor(
        base_model=base_model,
        img_feat_dim=IMG_FEAT_DIM,
        img_proj_dim=IMG_PROJ_DIM,
    )

    # Diagnostic 3b must freeze the complete baseline predictor.
    baseline_trainable = [
        name for name, p in model.named_parameters()
        if p.requires_grad and (
            name.startswith('gru.')
            or name.startswith('time_proj.')
            or name.startswith('base_head.')
        )
    ]
    if baseline_trainable:
        raise RuntimeError(
            'Diagnostic 3b baseline is not fully frozen. '
            f'Unexpected trainable parameters: {baseline_trainable}'
        )

    trainable_names = [name for name, p in model.named_parameters() if p.requires_grad]
    print('trainable parameters:')
    for name in trainable_names:
        print('  ', name)

    # Zero-initialised residual must reproduce the seed-matched baseline exactly.
    check_batch = next(iter(val_loader))
    cgm, tsm, has_meal, img_feat, _ = check_batch

    model.eval()
    base_model.eval()

    with torch.no_grad():
        base_pred = base_model(cgm, tsm, has_meal)
        residual_pred = model(cgm, tsm, img_feat, has_meal)

    init_diff = (base_pred - residual_pred).abs().max().item()
    print(f'initial max |residual - baseline| = {init_diff:.8f}')

    if init_diff > 1e-5:
        raise RuntimeError(
            f'seed={seed}: residual model does not match the frozen baseline '
            f'at initialisation (max diff={init_diff}).'
        )

    model, history = train.train_model(
        model,
        train_loader,
        val_loader,
        stats['cgm_mean'],
        stats['cgm_std'],
        lr=1e-4,
        weight_decay=1e-5,
        forward_fn=forward_fn,
        checkpoint_dir=(
            EXPERIMENT_DIR / 'checkpoints'
            / f'diagnostic3b_resnet18_dim4_seed{seed}'
        ),
        verbose=False,
        progress_bar=True,
        progress_desc=f'D3b seed={seed}',
    )

    y_pred_val, y_true_val = train.predict(
        model,
        val_loader,
        forward_fn=forward_fn,
    )
    val_results = metrics.evaluate(
        y_pred_val,
        y_true_val,
        stats['cgm_mean'],
        stats['cgm_std'],
    )

    seed_result = {
        'model': MODEL_NAME,
        'seed': seed,
        'encoder': ENCODER_NAME,
        'img_feat_dim': IMG_FEAT_DIM,
        'img_proj_dim': IMG_PROJ_DIM,
        'best_epoch': len(history['train_loss']),
        'initial_max_abs_diff_vs_baseline': init_diff,
        'val_results': val_results,
    }
    all_results.append(seed_result)

    result_path = RESULTS_DIR / f'diagnostic3b_resnet18_dim4_seed{seed}_results.json'
    with open(result_path, 'w') as f:
        json.dump(seed_result, f, indent=2)

    print(f'best epoch: {seed_result["best_epoch"]}')
    for h in HORIZONS:
        entry = val_results.get(str(h), val_results.get(h))
        print(f'  {h:>3}-min validation RMSE = {entry["rmse"]:.4f}')

print(f'\nCompleted Diagnostic 3b for {len(all_results)} seeds.')



------------------ seed=7 ------------------
trainable parameters:
   img_proj.0.weight
   img_proj.0.bias
   image_head.0.weight
   image_head.0.bias
   image_head.3.weight
   image_head.3.bias
initial max |residual - baseline| = 0.00000000
device: cuda (NVIDIA A100-SXM4-40GB)  model: FrozenImageOnlyResidualPredictor  params: 1,028,478


D3b seed=7:   0%|          | 0/50 [00:00<?, ?it/s]

best epoch: 13
   15-min validation RMSE = 10.6191
   30-min validation RMSE = 16.6756
   60-min validation RMSE = 23.9898
   90-min validation RMSE = 28.2390
  120-min validation RMSE = 30.9529

------------------ seed=14 ------------------
trainable parameters:
   img_proj.0.weight
   img_proj.0.bias
   image_head.0.weight
   image_head.0.bias
   image_head.3.weight
   image_head.3.bias
initial max |residual - baseline| = 0.00000000
device: cuda (NVIDIA A100-SXM4-40GB)  model: FrozenImageOnlyResidualPredictor  params: 1,028,478


D3b seed=14:   0%|          | 0/50 [00:00<?, ?it/s]

best epoch: 12
   15-min validation RMSE = 10.4966
   30-min validation RMSE = 16.6465
   60-min validation RMSE = 24.0507
   90-min validation RMSE = 28.3378
  120-min validation RMSE = 31.1094

------------------ seed=21 ------------------
trainable parameters:
   img_proj.0.weight
   img_proj.0.bias
   image_head.0.weight
   image_head.0.bias
   image_head.3.weight
   image_head.3.bias
initial max |residual - baseline| = 0.00000000
device: cuda (NVIDIA A100-SXM4-40GB)  model: FrozenImageOnlyResidualPredictor  params: 1,028,478


...

best epoch: 14
   15-min validation RMSE = 10.5591
   30-min validation RMSE = 16.7172
   60-min validation RMSE = 23.9967
   90-min validation RMSE = 28.2414
  120-min validation RMSE = 31.0239

------------------ seed=63 ------------------
trainable parameters:
   img_proj.0.weight
   img_proj.0.bias
   image_head.0.weight
   image_head.0.bias
   image_head.3.weight
   image_head.3.bias
initial max |residual - baseline| = 0.00000000
device: cuda (NVIDIA A100-SXM4-40GB)  model: FrozenImageOnlyResidualPredictor  params: 1,028,478


D3b seed=63:   0%|          | 0/50 [00:00<?, ?it/s]

best epoch: 14
   15-min validation RMSE = 10.5702
   30-min validation RMSE = 16.7073
   60-min validation RMSE = 23.9626
   90-min validation RMSE = 28.2964
  120-min validation RMSE = 31.1317

------------------ seed=70 ------------------
trainable parameters:
   img_proj.0.weight
   img_proj.0.bias
   image_head.0.weight
   image_head.0.bias
   image_head.3.weight
   image_head.3.bias
initial max |residual - baseline| = 0.00000000
device: cuda (NVIDIA A100-SXM4-40GB)  model: FrozenImageOnlyResidualPredictor  params: 1,028,478


D3b seed=70:   0%|          | 0/50 [00:00<?, ?it/s]

best epoch: 14
   15-min validation RMSE = 10.6377
   30-min validation RMSE = 16.7160
   60-min validation RMSE = 23.9873
   90-min validation RMSE = 28.2780
  120-min validation RMSE = 31.1261

Completed Diagnostic 3b for 10 seeds.


In [6]:
def _horizon_entry(result_dict, horizon):
    if str(horizon) in result_dict:
        return result_dict[str(horizon)]
    if horizon in result_dict:
        return result_dict[horizon]
    raise KeyError(
        f'Horizon {horizon} not found. Available keys: {list(result_dict.keys())}'
    )


# Aggregate Diagnostic 3b validation RMSE across seeds.
d3b_by_h = {
    h: np.asarray(
        [
            float(_horizon_entry(r['val_results'], h)['rmse'])
            for r in all_results
        ],
        dtype=float,
    )
    for h in HORIZONS
}

summary = {
    str(h): {
        'rmse_mean': float(np.mean(d3b_by_h[h])),
        'rmse_sd_across_seeds': float(np.std(d3b_by_h[h], ddof=1)),
    }
    for h in HORIZONS
}

print('=== Diagnostic 3b validation RMSE across seeds ===')
for h in HORIZONS:
    print(
        f'{h:>3}-min: '
        f'{summary[str(h)]["rmse_mean"]:.3f} +/- '
        f'{summary[str(h)]["rmse_sd_across_seeds"]:.3f}'
    )

with open(SUMMARY_PATH, 'w') as f:
    json.dump(
        {
            'model': MODEL_NAME,
            'seeds': SEEDS,
            'n_seeds': len(SEEDS),
            'img_proj_dim': IMG_PROJ_DIM,
            'validation_summary': summary,
        },
        f,
        indent=2,
    )

print(f'\nSaved summary to: {SUMMARY_PATH}')


=== Diagnostic 3b validation RMSE across seeds ===
 15-min: 10.568 +/- 0.067
 30-min: 16.670 +/- 0.056
 60-min: 23.985 +/- 0.072
 90-min: 28.277 +/- 0.066
120-min: 31.076 +/- 0.074

Saved summary to: /content/drive/MyDrive/dissertation/experiments/4_diagnostic/results/diagnostic_3b/diagnostic3b_resnet18_dim4_validation_summary.json


In [15]:
# Paired validation comparison:
#   CGM+Time baseline vs Diagnostic 3a vs Diagnostic 3b
#
# IMPORTANT: validation results only. No test metrics are loaded.

BASE_DIR = (
    PROJECT_ROOT / 'experiments' / '3_multimodal' / 'multimodal' / 'results'
    / 'gated_cgm_time'
)

D3A_DIR = (
    PROJECT_ROOT / 'experiments' / '4_diagnostic'/ 'results' / 'diagnostic_3_resnet'
)


def load_seed_results(result_dir, stub, seeds, result_key='val_results'):
    loaded = {}

    for seed in seeds:
        path = result_dir / f'{stub}_seed{seed}_results.json'

        if not path.exists():
            raise FileNotFoundError(
                f'Missing result file for seed {seed}: {path}\n'
                'Adjust result_dir/stub below if your saved Diagnostic 3a filenames differ.'
            )

        with open(path) as f:
            result = json.load(f)

        if result_key not in result:
            raise KeyError(
                f'{path.name} does not contain "{result_key}". '
                f'Available keys: {list(result.keys())}'
            )

        loaded[seed] = result

    return loaded


def rmse_by_seed(loaded_results, seeds, horizons, result_key='val_results'):
    return {
        h: np.asarray(
            [
                float(_horizon_entry(loaded_results[seed][result_key], h)['rmse'])
                for seed in seeds
            ],
            dtype=float,
        )
        for h in horizons
    }


# Baseline filenames used in the existing multimodal experiments.
base_loaded = load_seed_results(
    BASE_DIR,
    'gated_cgm_time',
    SEEDS,
)

# Try the likely Diagnostic 3a locations/names.
d3a_candidates = [
    (
        PROJECT_ROOT / 'experiments' / '4_diagnostic' / 'results'
        / 'diagnostic_3_resnet',
        'diagnostic3_partial_freeze',
    )
]

d3a_loaded = None
last_error = None
for d3a_dir, d3a_stub in d3a_candidates:
    try:
        d3a_loaded = load_seed_results(d3a_dir, d3a_stub, SEEDS)
        print(f'Loaded Diagnostic 3a from: {d3a_dir} / {d3a_stub}')
        break
    except FileNotFoundError as exc:
        last_error = exc

if d3a_loaded is None:
    raise FileNotFoundError(
        'Could not locate Diagnostic 3a per-seed result files. '
        'Edit d3a_candidates to match your actual saved path/stub.\n'
        f'Last attempted error: {last_error}'
    )

base_by_h = rmse_by_seed(base_loaded, SEEDS, HORIZONS)
d3a_by_h = rmse_by_seed(d3a_loaded, SEEDS, HORIZONS)


print('=== Paired validation RMSE across seeds ===')
print(
    f"{'horizon':>9}"
    f"{'CGM+time':>19}"
    f"{'D3a partial freeze':>24}"
    f"{'D3b residual':>21}"
)

for h in HORIZONS:
    base = base_by_h[h]
    d3a = d3a_by_h[h]
    d3b = d3b_by_h[h]

    print(
        f"{h:>6} min"
        f"{np.mean(base):>10.3f} +/- {np.std(base, ddof=1):<6.3f}"
        f"{np.mean(d3a):>15.3f} +/- {np.std(d3a, ddof=1):<6.3f}"
        f"{np.mean(d3b):>12.3f} +/- {np.std(d3b, ddof=1):<6.3f}"
    )


def print_paired_effect(title, reference_by_h, candidate_by_h, candidate_label):
    print(f'\n=== {title} ===')
    print(
        f"{'horizon':>9}"
        f"{'delta RMSE':>22}"
        f"{'improvement %':>24}"
        f"{candidate_label:>20}"
    )

    for h in HORIZONS:
        ref = reference_by_h[h]
        cand = candidate_by_h[h]

        # Negative delta = candidate has lower RMSE.
        delta = cand - ref
        improvement_pct = (ref - cand) / ref * 100.0

        wins = int(np.sum(cand < ref))
        ties = int(np.sum(np.isclose(cand, ref, atol=1e-12)))
        losses = len(SEEDS) - wins - ties

        print(
            f"{h:>6} min"
            f"{np.mean(delta):>11.4f} +/- {np.std(delta, ddof=1):<7.4f}"
            f"{np.mean(improvement_pct):>13.3f} +/- "
            f"{np.std(improvement_pct, ddof=1):<7.3f}"
            f"{wins:>8}/{len(SEEDS)}"
            f"  (ties={ties}, losses={losses})"
        )


print_paired_effect(
    'Diagnostic 3b paired effect: Residual vs CGM+Time',
    base_by_h,
    d3b_by_h,
    'D3b better seeds',
)

print_paired_effect(
    'Diagnostic 3b vs Diagnostic 3a',
    d3a_by_h,
    d3b_by_h,
    'D3b better seeds',
)


rows = []
for h in HORIZONS:
    for i, seed in enumerate(SEEDS):
        base = base_by_h[h][i]
        d3a = d3a_by_h[h][i]
        d3b = d3b_by_h[h][i]

        rows.append({
            'horizon_min': h,
            'seed': seed,
            'cgm_time_rmse': base,
            'diagnostic3a_partial_freeze_rmse': d3a,
            'diagnostic3b_residual_rmse': d3b,
            'd3b_minus_base': d3b - base,
            'd3b_minus_d3a': d3b - d3a,
            'd3b_improvement_vs_base_pct': (base - d3b) / base * 100.0,
            'd3b_improvement_vs_d3a_pct': (d3a - d3b) / d3a * 100.0,
        })

# paired_validation_df = pd.DataFrame(rows)

# print('\n=== Per-seed paired validation detail ===')
# display(paired_validation_df.round(4))

# paired_validation_df.to_csv(PAIRED_PATH, index=False)
# print(f'\nSaved paired validation table to: {PAIRED_PATH}')


Loaded Diagnostic 3a from: /content/drive/MyDrive/dissertation/experiments/4_diagnostic/results/diagnostic_3_resnet / diagnostic3_partial_freeze
=== Paired validation RMSE across seeds ===
  horizon           CGM+time      D3a partial freeze         D3b residual
    15 min    10.580 +/- 0.073          10.560 +/- 0.035       10.568 +/- 0.067 
    30 min    16.705 +/- 0.060          16.601 +/- 0.064       16.670 +/- 0.056 
    60 min    24.087 +/- 0.063          23.916 +/- 0.068       23.985 +/- 0.072 
    90 min    28.415 +/- 0.047          28.374 +/- 0.075       28.277 +/- 0.066 
   120 min    31.270 +/- 0.050          31.221 +/- 0.083       31.076 +/- 0.074 

=== Diagnostic 3b paired effect: Residual vs CGM+Time ===
  horizon            delta RMSE           improvement %    D3b better seeds
    15 min    -0.0118 +/- 0.0084         0.112 +/- 0.078        10/10  (ties=0, losses=0)
    30 min    -0.0348 +/- 0.0095         0.208 +/- 0.056        10/10  (ties=0, losses=0)
    60 min    -0.